# CELL 1 — Install packages

In [ ]:
!pip install datasets transformers accelerate scikit-learn -q
print("✅ Packages installed")

✅ Packages installed


# CELL 2 — Mount Google Drive

In [ ]:
# Fix the path
DRIVE_PATH  = '/content/drive/MyDrive/PoliGraphX (1)'
SPLITS_PATH = '/content/drive/MyDrive/PoliGraphX (1)/splits'

import os
os.makedirs(SPLITS_PATH, exist_ok=True)

csv_path = os.path.join(DRIVE_PATH, 'cleaned_tweets.csv')
if os.path.exists(csv_path):
    print(f"✅ cleaned_tweets.csv FOUND")
    print(f"   Path: {csv_path}")
else:
    print(f"❌ Still not found")

✅ cleaned_tweets.csv FOUND
   Path: /content/drive/MyDrive/PoliGraphX (1)/cleaned_tweets.csv


In [ ]:
import os

# Search for the file anywhere in your drive
print("Searching for cleaned_tweets.csv in your Drive...")
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if 'cleaned' in file.lower() or 'tweets' in file.lower():
            print(f"FOUND: {os.path.join(root, file)}")

# Also list what's in MyDrive root
print("\nFolders in MyDrive:")
for item in os.listdir('/content/drive/MyDrive'):
    print(f"  '{item}'")

Searching for cleaned_tweets.csv in your Drive...
FOUND: /content/drive/MyDrive/PoliGraphX (1)/cleaned_tweets.csv

Folders in MyDrive:
  '20251206_123002.jpg'
  'Classroom'
  'Lab_Aug_18.pdf'
  'Lab_Aug_18.gdoc'
  'DS - Data structures - problem sheet 1 (1).gdoc'
  'DS - Data structures - problem sheet 1.gdoc'
  'DS - Data structures - problem sheet 1.docx'
  'OCLabProblemsheet2.docx'
  '4 Control Structures - Part 1.gdoc'
  '789.gdoc'
  'main.cpp'
  'Colab Notebooks'
  'OilTank.h'
  'binary search tree.c'
  'drone.cpp'
  'drone.h'
  'buses.txt'
  'Atm no .gdoc'
  '23XD37-ADS Problem Sheet 1.docx'
  'Interpolation - I.gdoc'
  'Untitled document (2).gdoc'
  'Fassal- Farhana.gdoc'
  'Interpolation - I tutorial test-23PD05.docx'
  'certificate-d67fd4b0-a6cd-4219-aa81-640494396818.pdf'
  'CHI SQUARE_.docx'
  'OS_Lab_Manual.pdf'
  'DV-worksheet2.docx'
  'OS_Lab_Manual (1).gdoc'
  'OS_Lab_Manual.gdoc'
  '35.docx'
  'DV-worksheet5-Correlation  and Regression.docx'
  'Lab_PracticeTest1.docx'
 

# CELL 3 — Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import time
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import pipeline
from sklearn.model_selection import train_test_split

# Paths
DRIVE_PATH  = '/content/drive/MyDrive/PoliGraphX (1)'
SPLITS_PATH = os.path.join(DRIVE_PATH, 'splits')
os.makedirs(SPLITS_PATH, exist_ok=True)

# Label encoding
LABEL2ID = {'Left': 0, 'Right': 1, 'Neutral': 2}
ID2LABEL = {0: 'Left', 1: 'Right', 2: 'Neutral'}

# Targets per class
TARGET_LEFT    = 1200
TARGET_RIGHT   = 1200
TARGET_NEUTRAL = 800

RANDOM_SEED    = 42

print("✅ Imports done")
print(f"   Drive path  : {DRIVE_PATH}")
print(f"   Splits path : {SPLITS_PATH}")


✅ Imports done
   Drive path  : /content/drive/MyDrive/PoliGraphX (1)
   Splits path : /content/drive/MyDrive/PoliGraphX (1)/splits


# CELL 4 — Helper: clean text

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|https\S+|www\S+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("✅ Helper functions ready")

✅ Helper functions ready


# CELL 5 — Load Senator Tweets (Left + Right labels)

In [ ]:
print("Loading senator tweets from HuggingFace...")
print("This gives us real Left/Right labeled tweets")
print()

senator_ds = load_dataset("m-newhauser/senator-tweets", split="train")
senator_df = senator_ds.to_pandas()

print(f"Loaded       : {len(senator_df):,} senator tweets")
print(f"Columns      : {list(senator_df.columns)}")
print(f"Parties      : {senator_df['party'].unique()}")

# Map Democrat → Left, Republican → Right
senator_df['ideology'] = senator_df['party'].map({
    'Democrat':   'Left',
    'Republican': 'Right'
})

# Clean text
senator_df['text'] = senator_df['text'].astype(str).apply(clean_text)

# Keep valid rows
senator_df = senator_df[
    senator_df['ideology'].isin(['Left', 'Right']) &
    (senator_df['text'].str.len() >= 15)
].copy()

print(f"After clean  : {len(senator_df):,} tweets")
print(f"Distribution : {senator_df['ideology'].value_counts().to_dict()}")

# Sample balanced Left / Right
left_df  = senator_df[senator_df['ideology'] == 'Left'].sample(n=TARGET_LEFT,  random_state=RANDOM_SEED)
right_df = senator_df[senator_df['ideology'] == 'Right'].sample(n=TARGET_RIGHT, random_state=RANDOM_SEED)

df_senator           = pd.concat([left_df, right_df])[['text', 'ideology']].copy()
df_senator['source'] = 'senator_huggingface'

print(f"\n✅ Senator dataset ready")
print(f"   Left  : {len(left_df):,}")
print(f"   Right : {len(right_df):,}")
print(f"   Total : {len(df_senator):,}")

Loading senator tweets from HuggingFace...
This gives us real Left/Right labeled tweets



README.md:   0%|          | 0.00/659 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/186M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/46.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/79754 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19939 [00:00<?, ? examples/s]

Loaded       : 79,754 senator tweets
Columns      : ['date', 'id', 'username', 'text', 'party', 'labels', 'embeddings']
Parties      : ['Democrat' 'Republican']
After clean  : 78,607 tweets
Distribution : {'Left': 40548, 'Right': 38059}

✅ Senator dataset ready
   Left  : 1,200
   Right : 1,200
   Total : 2,400



# CELL 6 — Load YOUR tweets + Auto-label as Neutral

In [ ]:
print("Loading your cleaned_tweets.csv...")

csv_path = os.path.join(DRIVE_PATH, 'cleaned_tweets.csv')
df_yours = pd.read_csv(csv_path, low_memory=False)
df_yours = df_yours[df_yours['clean_text'].notna()].copy()
df_yours = df_yours[df_yours['clean_text'].str.len() >= 15].copy()
df_yours = df_yours.reset_index(drop=True)

print(f"Your tweets  : {len(df_yours):,}")

# Sample for zero-shot labeling
SAMPLE_SIZE = 3000
df_sample   = df_yours.sample(n=min(SAMPLE_SIZE, len(df_yours)), random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Sample size  : {len(df_sample):,} tweets for zero-shot")

# ── Load zero-shot model
# On Colab with GPU this is much faster
print("\nLoading facebook/bart-large-mnli...")
import torch
device = 0 if torch.cuda.is_available() else -1
print(f"Device       : {'GPU ✅' if device == 0 else 'CPU'}")

CANDIDATE_LABELS = [
    "left-wing liberal politics democrat",
    "right-wing conservative politics republican",
    "neutral unrelated not political"
]
LABEL_MAP = {
    "left-wing liberal politics democrat":         "Left",
    "right-wing conservative politics republican": "Right",
    "neutral unrelated not political":             "Neutral"
}

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)
print("✅ Model loaded")

Loading your cleaned_tweets.csv...
Your tweets  : 40,507
Sample size  : 3,000 tweets for zero-shot

Loading facebook/bart-large-mnli...
Device       : GPU ✅


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Model loaded


# CELL 7 — Run Zero-Shot Classification

In [ ]:
print(f"Running zero-shot on {len(df_sample):,} tweets...")
print("On GPU this takes ~5-10 minutes")
print()

BATCH_SIZE = 32 if torch.cuda.is_available() else 8
texts      = df_sample['clean_text'].str[:280].tolist()
labels_out = []
scores_out = []

t_start = time.time()

for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i: i + BATCH_SIZE]

    try:
        results = classifier(batch, candidate_labels=CANDIDATE_LABELS, multi_label=False)
        if isinstance(results, dict):
            results = [results]
        for res in results:
            labels_out.append(LABEL_MAP.get(res['labels'][0], 'Neutral'))
            scores_out.append(round(res['scores'][0], 4))
    except Exception as e:
        for _ in batch:
            labels_out.append('Neutral')
            scores_out.append(0.0)

    done    = min(i + BATCH_SIZE, len(texts))
    elapsed = time.time() - t_start
    eta     = (len(texts) - done) / (done / elapsed) / 60 if done > 0 else 0
    print(f"  {done}/{len(texts)} | {done/len(texts)*100:.1f}% | ETA: {eta:.1f} min", end='\r')

print(f"\n✅ Done in {(time.time()-t_start)/60:.1f} minutes")

df_sample['predicted'] = labels_out
df_sample['confidence'] = scores_out

# Show distribution
print("\nZero-shot results:")
for label in ['Left', 'Right', 'Neutral']:
    n = (df_sample['predicted'] == label).sum()
    print(f"  {label:<10}: {n:,} ({n/len(df_sample)*100:.1f}%)")

# Keep only high-confidence Neutral tweets
df_neutral = df_sample[
    (df_sample['predicted'] == 'Neutral') &
    (df_sample['confidence'] >= 0.85)
].copy()

print(f"\nHigh-confidence Neutral (>=0.85): {len(df_neutral):,}")

df_neutral_sample           = df_neutral.sample(
    n=min(TARGET_NEUTRAL, len(df_neutral)), random_state=RANDOM_SEED
)[['clean_text']].copy()
df_neutral_sample.columns   = ['text']
df_neutral_sample['ideology'] = 'Neutral'
df_neutral_sample['source']   = 'your_tweets_autolabeled'

print(f"Selected Neutral tweets: {len(df_neutral_sample):,}")

# Free GPU memory
del classifier
import gc; gc.collect()
torch.cuda.empty_cache()
print("✅ Memory cleared")

Running zero-shot on 3,000 tweets...
On GPU this takes ~5-10 minutes



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  3000/3000 | 100.0% | ETA: 0.0 min
✅ Done in 3.7 minutes

Zero-shot results:
  Left      : 803 (26.8%)
  Right     : 1,566 (52.2%)
  Neutral   : 631 (21.0%)

High-confidence Neutral (>=0.85): 38
Selected Neutral tweets: 38
✅ Memory cleared


# CELL 8 — Combine + Create Train/Val/Test Splits

In [ ]:
print("Combining datasets...")

df_combined           = pd.concat([df_senator, df_neutral_sample], ignore_index=True)
df_combined           = df_combined.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
df_combined['label']  = df_combined['ideology'].map(LABEL2ID)
df_combined['text']   = df_combined['text'].astype(str).str[:512]
df_combined           = df_combined[df_combined['text'].notna() & df_combined['label'].notna()].copy()

print(f"\nCombined dataset: {len(df_combined):,} tweets")
print("Distribution:")
for label, count in df_combined['ideology'].value_counts().items():
    print(f"  {label:<10}: {count:,} ({count/len(df_combined)*100:.1f}%)")

# ── 80 / 10 / 10 stratified split
df_train, df_temp = train_test_split(
    df_combined, test_size=0.20,
    stratify=df_combined['label'], random_state=RANDOM_SEED
)
df_val, df_test = train_test_split(
    df_temp, test_size=0.50,
    stratify=df_temp['label'], random_state=RANDOM_SEED
)

print(f"\nSplit sizes:")
print(f"  Train : {len(df_train):,}")
print(f"  Val   : {len(df_val):,}")
print(f"  Test  : {len(df_test):,}")

# ── Save to Google Drive
SAVE_COLS = ['text', 'label', 'ideology']

train_path = os.path.join(SPLITS_PATH, 'train.csv')
val_path   = os.path.join(SPLITS_PATH, 'val.csv')
test_path  = os.path.join(SPLITS_PATH, 'test.csv')

df_train[SAVE_COLS].reset_index(drop=True).to_csv(train_path, index=False)
df_val[SAVE_COLS].reset_index(drop=True).to_csv(val_path,     index=False)
df_test[SAVE_COLS].reset_index(drop=True).to_csv(test_path,   index=False)

print(f"\n✅ Files saved to Google Drive:")
print(f"   → {train_path}")
print(f"   → {val_path}")
print(f"   → {test_path}")
print(f"\n✅ NOTEBOOK 1 COMPLETE — Open Notebook 2 to fine-tune models")

Combining datasets...

Combined dataset: 2,438 tweets
Distribution:
  Left      : 1,200 (49.2%)
  Right     : 1,200 (49.2%)
  Neutral   : 38 (1.6%)

Split sizes:
  Train : 1,950
  Val   : 244
  Test  : 244

✅ Files saved to Google Drive:
   → /content/drive/MyDrive/PoliGraphX (1)/splits/train.csv
   → /content/drive/MyDrive/PoliGraphX (1)/splits/val.csv
   → /content/drive/MyDrive/PoliGraphX (1)/splits/test.csv

✅ NOTEBOOK 1 COMPLETE — Open Notebook 2 to fine-tune models


#Removing Neutral due to Polarization

In [ ]:
# QUICK FIX — Remove Neutral, keep only Left vs Right
import pandas as pd
import os

SPLITS_PATH = '/content/drive/MyDrive/PoliGraphX (1)/splits'
LABEL2ID    = {'Left': 0, 'Right': 1}

for split in ['train', 'val', 'test']:
    path = os.path.join(SPLITS_PATH, f'{split}.csv')
    df   = pd.read_csv(path)
    df   = df[df['ideology'].isin(['Left', 'Right'])].copy()
    df['label'] = df['ideology'].map(LABEL2ID)
    df.reset_index(drop=True).to_csv(path, index=False)
    print(f"{split}: {len(df):,} tweets | {df['ideology'].value_counts().to_dict()}")

print("\n✅ Splits updated — Ready for Notebook 2")

train: 1,920 tweets | {'Left': 960, 'Right': 960}
val: 240 tweets | {'Right': 120, 'Left': 120}
test: 240 tweets | {'Right': 120, 'Left': 120}

✅ Splits updated — Ready for Notebook 2


#Replacement for cell 5 ( increasing the training dataset)

In [ ]:
# INCREASE TRAINING DATA — 3000 per class instead of 1200
TARGET_LEFT  = 3000
TARGET_RIGHT = 3000
LABEL2ID     = {'Left': 0, 'Right': 1}
RANDOM_SEED  = 42

left_df  = senator_df[senator_df['ideology'] == 'Left'].sample(n=TARGET_LEFT,  random_state=RANDOM_SEED)
right_df = senator_df[senator_df['ideology'] == 'Right'].sample(n=TARGET_RIGHT, random_state=RANDOM_SEED)

df_senator           = pd.concat([left_df, right_df])[['text', 'ideology']].copy()
df_senator['source'] = 'senator_huggingface'
df_senator['label']  = df_senator['ideology'].map(LABEL2ID)
df_senator['text']   = df_senator['text'].astype(str).str[:512]
df_senator           = df_senator.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"New dataset size : {len(df_senator):,} tweets")
print(f"Distribution     : {df_senator['ideology'].value_counts().to_dict()}")

# Re-split 80/10/10
from sklearn.model_selection import train_test_split

df_train, df_temp = train_test_split(df_senator, test_size=0.20, stratify=df_senator['label'], random_state=RANDOM_SEED)
df_val, df_test   = train_test_split(df_temp,    test_size=0.50, stratify=df_temp['label'],    random_state=RANDOM_SEED)

SAVE_COLS = ['text', 'label', 'ideology']
df_train[SAVE_COLS].reset_index(drop=True).to_csv(f'{SPLITS_PATH}/train.csv', index=False)
df_val[SAVE_COLS].reset_index(drop=True).to_csv(f'{SPLITS_PATH}/val.csv',     index=False)
df_test[SAVE_COLS].reset_index(drop=True).to_csv(f'{SPLITS_PATH}/test.csv',   index=False)

print(f"\nSplit sizes:")
print(f"  Train : {len(df_train):,} tweets")
print(f"  Val   : {len(df_val):,} tweets")
print(f"  Test  : {len(df_test):,} tweets")
print(f"\n✅ Updated splits saved to Google Drive")


New dataset size : 6,000 tweets
Distribution     : {'Left': 3000, 'Right': 3000}

Split sizes:
  Train : 4,800 tweets
  Val   : 600 tweets
  Test  : 600 tweets

✅ Updated splits saved to Google Drive
